# 🚗 Car Price Prediction with Machine Learning

Predicting the **selling price of used cars** using regression models, based on
features such as brand, age, mileage (km driven), fuel type, and transmission.

**Dataset:** CAR DETAILS FROM CAR DEKHO (Kaggle) — 4,340 used car listings scraped
from CarDekho.com, covering car name, manufacturing year, selling price, km driven,
fuel type, seller type, transmission, and ownership history.

**Tech stack:** Python · pandas · scikit-learn · matplotlib · seaborn

**Notebook runs end-to-end in Google Colab** — no local files or paths needed;
the dataset is pulled directly from a public GitHub mirror of the Kaggle dataset.

---

**Workflow**
1. Load data
2. Data cleaning
3. Feature engineering
4. Exploratory Data Analysis (EDA)
5. Encoding categorical variables
6. Correlation heatmap
7. Train/test split
8. Model training (Linear Regression, Random Forest, Gradient Boosting)
9. Model evaluation (MAE, RMSE, R²)
10. Feature importance


## 1. Setup & Imports

In [ ]:
# Install (Colab already has these, but this keeps the notebook self-contained)
!pip install -q pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
print("Libraries loaded.")


## 2. Load Dataset

We use the **"CAR DETAILS FROM CAR DEKHO"** dataset (the same one referenced on
Kaggle: https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho).

To make this notebook run anywhere with zero setup (including Colab), we load it
directly from a public raw-CSV mirror of the dataset on GitHub. If you'd rather use
your own copy, upload it in Colab (`Files` panel → upload) and just change the
`DATA_URL` line below to `"car_dekho.csv"`.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/chandanverma07/DataSets/master/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv"

df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 3. Data Cleaning

Steps:
- Check and handle missing values
- Remove duplicate rows
- Standardize inconsistent categorical text (e.g. `"Petrol"` vs `"petrol"`)


In [ ]:
# --- Missing values ---
print("Missing values per column:\n")
print(df.isnull().sum())


In [ ]:
# If any nulls exist, drop rows with nulls in critical columns (name, year, selling_price)
# and impute/​drop others as appropriate. This dataset is generally clean, but we handle it robustly.
critical_cols = ["name", "year", "selling_price", "km_driven"]
df = df.dropna(subset=critical_cols)

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:\n")
print(df.isnull().sum())


In [ ]:
# --- Duplicates ---
print("Duplicate rows before:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after:", df.duplicated().sum())
print("Shape after dropping duplicates:", df.shape)


In [ ]:
# --- Standardize inconsistent categorical values ---
# Strip whitespace + normalize casing to Title Case for all text/categorical columns
cat_cols = ["name", "fuel", "seller_type", "transmission", "owner"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

for col in ["fuel", "seller_type", "transmission", "owner"]:
    df[col] = df[col].str.title()   # e.g. "petrol"/"PETROL"/"Petrol " -> "Petrol"

print("Unique fuel values:", df["fuel"].unique())
print("Unique seller_type values:", df["seller_type"].unique())
print("Unique transmission values:", df["transmission"].unique())
print("Unique owner values:", df["owner"].unique())


In [ ]:
# --- Sanity-check numeric ranges (basic outlier/consistency guard) ---
CURRENT_YEAR = 2026
print("Year range:", df["year"].min(), "-", df["year"].max())
print("Selling price range:", df["selling_price"].min(), "-", df["selling_price"].max())
print("Km driven range:", df["km_driven"].min(), "-", df["km_driven"].max())

# Drop any impossible rows (year in the future, non-positive price/km)
df = df[(df["year"] <= CURRENT_YEAR) & (df["selling_price"] > 0) & (df["km_driven"] >= 0)].reset_index(drop=True)
print("Shape after sanity checks:", df.shape)


## 4. Feature Engineering

- **`car_age`** — derived from the manufacturing `year`
- **`brand`** — extracted from the `name` column (first word), with a few known
  two-word brand names (e.g. *Land Rover*, *Maruti Suzuki*) merged correctly


In [ ]:
CURRENT_YEAR = 2026
df["car_age"] = CURRENT_YEAR - df["year"]

# A few brands are legitimately two words — check for these before taking the first token
two_word_brands = ["Land Rover", "Maruti Suzuki", "Mahindra Renault", "Force Motors", "Isuzu D-Max"]

def extract_brand(name):
    for brand in two_word_brands:
        if name.startswith(brand):
            return brand
    return name.split(" ")[0]

df["brand"] = df["name"].apply(extract_brand)

# A couple of brand-name aliases that show up in used-car listings
brand_aliases = {
    "Maruti": "Maruti",
    "OpelCorsa": "Opel",
}
df["brand"] = df["brand"].replace(brand_aliases)

print("Number of unique brands:", df["brand"].nunique())
df[["name", "year", "car_age", "brand"]].head(10)


In [ ]:
df["brand"].value_counts()


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# --- Distribution of selling prices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["selling_price"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Selling Price")
axes[0].set_xlabel("Selling Price (INR)")

sns.histplot(np.log1p(df["selling_price"]), bins=50, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of log(Selling Price)")
axes[1].set_xlabel("log(1 + Selling Price)")

plt.tight_layout()
plt.show()


In [ ]:
# --- Price vs Fuel Type ---
plt.figure(figsize=(9, 5))
order = df.groupby("fuel")["selling_price"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="fuel", y="selling_price", order=order, palette="Set2")
plt.title("Selling Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Selling Price (INR)")
plt.yscale("log")
plt.tight_layout()
plt.show()


In [ ]:
# --- Price vs Car Age ---
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="car_age", y="selling_price", hue="fuel", alpha=0.6, s=30)
plt.title("Selling Price vs Car Age")
plt.xlabel("Car Age (years)")
plt.ylabel("Selling Price (INR)")
plt.yscale("log")
plt.legend(title="Fuel", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# --- A few extra views: transmission, top brands ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="transmission", y="selling_price", ax=axes[0], palette="Set3")
axes[0].set_title("Selling Price by Transmission")
axes[0].set_yscale("log")

top_brands = df["brand"].value_counts().nlargest(10).index
sns.boxplot(data=df[df["brand"].isin(top_brands)], x="brand", y="selling_price", ax=axes[1], palette="coolwarm")
axes[1].set_title("Selling Price by Brand (Top 10 by Listing Count)")
axes[1].set_yscale("log")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 6. Encode Categorical Variables

- **`owner`** has a natural order (First → Second → ... → Test Drive Car), so we
  use **ordinal encoding**.
- **`fuel`, `seller_type`, `transmission`** are nominal → **One-Hot Encoding**.
- **`brand`** has many categories → One-Hot Encoding, grouping rare brands
  (fewer than 15 listings) into an `"Other"` bucket to avoid an overly sparse
  feature space.


In [ ]:
df_model = df.copy()

# --- Ordinal encode owner ---
owner_order = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth & Above Owner": 4,
    "Test Drive Car": 0,
}
df_model["owner_encoded"] = df_model["owner"].map(owner_order)
df_model["owner_encoded"] = df_model["owner_encoded"].fillna(df_model["owner_encoded"].median())

# --- Group rare brands into "Other" ---
brand_counts = df_model["brand"].value_counts()
rare_brands = brand_counts[brand_counts < 15].index
df_model["brand_grouped"] = df_model["brand"].apply(lambda b: "Other" if b in rare_brands else b)
print("Brands kept individually:", df_model["brand_grouped"].nunique() - 1, " | grouped into 'Other':", len(rare_brands))

# --- One-hot encode nominal categoricals ---
categorical_features = ["fuel", "seller_type", "transmission", "brand_grouped"]
df_encoded = pd.get_dummies(df_model, columns=categorical_features, drop_first=True)

# Drop columns we no longer need as raw text/identifiers
df_encoded = df_encoded.drop(columns=["name", "owner", "brand", "year"])

print("Final encoded shape:", df_encoded.shape)
df_encoded.head()


## 7. Feature Correlation Heatmap

In [ ]:
numeric_cols = ["selling_price", "km_driven", "car_age", "owner_encoded"]
corr = df_encoded[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True, linewidths=0.5)
plt.title("Correlation Heatmap (Core Numeric Features)")
plt.tight_layout()
plt.show()


In [ ]:
# Full correlation heatmap including one-hot encoded features (top correlated with target)
full_corr = df_encoded.corr(numeric_only=True)["selling_price"].sort_values(ascending=False)
plt.figure(figsize=(6, 10))
sns.heatmap(full_corr.to_frame(), annot=True, cmap="coolwarm", fmt=".2f", cbar=False)
plt.title("Correlation of All Features with Selling Price")
plt.tight_layout()
plt.show()


## 8. Train / Test Split

In [ ]:
X = df_encoded.drop(columns=["selling_price"])
y = df_encoded["selling_price"]

# Ensure all feature columns are numeric (bool -> int for one-hot columns)
X = X.astype({col: "int" for col in X.select_dtypes(include="bool").columns})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 9. Model Training

We train and compare three regression models:
1. **Linear Regression** — simple baseline
2. **Random Forest Regressor** — bagging ensemble, handles non-linearity well
3. **Gradient Boosting Regressor** — boosting ensemble, typically the strongest performer


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained.")


## 10. Model Evaluation — MAE, RMSE, R²

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2 Score": r2})

results_df = pd.DataFrame(results).sort_values("R2 Score", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# --- Visualize model comparison ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metrics = ["MAE", "RMSE", "R2 Score"]
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, metric, color in zip(axes, metrics, colors):
    sns.barplot(data=results_df, x="Model", y=metric, ax=ax, color=color)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"Best performing model: {best_model_name}")
print(results_df.iloc[0])


In [ ]:
# --- Actual vs Predicted for the best model ---
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.4, s=20, color="teal")
lims = [0, max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, "r--", linewidth=1.5, label="Perfect Prediction")
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title(f"Actual vs Predicted Selling Price — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Feature Importance (Best Model)

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
elif hasattr(best_model, "coef_"):
    importances = pd.Series(np.abs(best_model.coef_), index=X.columns).sort_values(ascending=False)
else:
    importances = pd.Series(dtype=float)

top_n = 15
plt.figure(figsize=(8, 7))
sns.barplot(x=importances.head(top_n).values, y=importances.head(top_n).index, color="mediumpurple")
plt.title(f"Top {top_n} Feature Importances — {best_model_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importances.head(top_n)


## 12. Conclusion

- The dataset was cleaned (nulls, duplicates, inconsistent category text),
  engineered with `car_age` and `brand`, and encoded for modeling.
- Three regression models were trained and compared using **MAE, RMSE, and R²**.
- Tree-based ensembles (**Random Forest** / **Gradient Boosting**) typically
  outperform plain Linear Regression on this dataset, since price depends on
  non-linear interactions between age, km driven, brand, and fuel type.
- The feature importance chart confirms that **car age, km driven, and brand**
  are the strongest predictors of used car selling price — consistent with
  real-world used-car pricing intuition.

**Possible next steps:** hyperparameter tuning (GridSearchCV / RandomizedSearchCV),
log-transforming the target for a better-behaved residual distribution, trying
XGBoost/LightGBM, and deploying the best model behind a simple Streamlit app.
